In [ ]:
import pandas as pd
import optuna
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Load data
def load_data(file_path):
    """Load data from a CSV file."""
    return pd.read_csv(file_path)

df = load_data("C:\\Users\\Graeme\\Documents\\github\\tsfwpt\\data\\dodgerloopday_long.csv")

# Feature engineering
df_features = df.copy()
df_features['time'] = df_features['time'].astype(int)
df_features['hour'] = (df_features['time'] * 5 // 60) % 24
df_features['day'] = (df_features['time'] * 5 // (60 * 24)) % 7  # Day of the week
df_features['is_weekend'] = df_features['day'].isin([5, 6]).astype(int)

# Sort for lag/lead feature creation
df_features = df_features.sort_values(by=['series_id', 'time'])

# Create lag and lead features
for lag in [1, 2, 3, 6, 12]:  # 5min, 10min, 15min, 30min, 1hr
    df_features[f'lag_{lag}'] = df_features.groupby('series_id')['value'].shift(lag)
    df_features[f'lead_{lag}'] = df_features.groupby('series_id')['value'].shift(-lag)

# Create rolling window features
for window in [3, 6, 12, 24, 288]:  # 15min, 30min, 1hr, 2hr, 1day
    df_features[f'rolling_mean_{window}'] = df_features.groupby('series_id')['value'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean())
    df_features[f'rolling_std_{window}'] = df_features.groupby('series_id')['value'].transform(
        lambda x: x.rolling(window=window, min_periods=1).std())

# Fill missing values in features
for col in df_features.columns:
    if col != 'value' and df_features[col].isna().any():
        df_features[col] = df_features.groupby('series_id')[col].transform(
            lambda x: x.fillna(method='ffill').fillna(method='bfill'))

# Identify categorical columns for CatBoost
cat_features = ['class', 'source']
for col in cat_features:
    if col in df_features.columns:
        df_features[col] = df_features[col].astype('category')

# Define feature columns
feature_cols = [col for col in df_features.columns if col not in 
               ['value', 'value_imputed', 'value_filled', 'series_id']]
cat_feature_indices = [feature_cols.index(col) for col in cat_features if col in feature_cols]

# Split data into training and missing subsets
df_train = df_features[df_features['value'].notna()]
df_missing = df_features[df_features['value'].isna()]
X_train = df_train[feature_cols]
y_train = df_train['value']

# Define Optuna objective function for hyperparameter tuning
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'loss_function': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'verbose': False
    }
    
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    
    for train_idx, val_idx in tscv.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        train_pool = Pool(X_fold_train, y_fold_train, cat_features=cat_feature_indices)
        val_pool = Pool(X_fold_val, y_fold_val, cat_features=cat_feature_indices)
        
        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, verbose=False)
        
        preds = model.predict(X_fold_val)
        rmse = np.sqrt(mean_squared_error(y_fold_val, preds))
        scores.append(rmse)
    
    return np.mean(scores)

# Run Optuna study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
best_params = study.best_params # {'iterations': 482, 'learning_rate': 0.19641246758129247, 'depth': 7, 'l2_leaf_reg': 2.5346992562109437e-07, 'border_count': 231}

# Train final model with best parameters
final_model = CatBoostRegressor(
    **best_params,
    loss_function='RMSE',
    task_type='GPU',
    devices='0',
    verbose=False
)
train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
final_model.fit(train_pool)

# Impute missing values
X_missing = df_missing[feature_cols]
df_features.loc[df_missing.index, 'value_imputed'] = final_model.predict(X_missing)
df_features['value_filled'] = df_features['value'].copy()
df_features.loc[df_features['value'].isna(), 'value_filled'] = df_features['value_imputed']

# Iterative refinement of imputed values
prev_imputed_values = df_features.loc[df_missing.index, 'value_imputed'].copy()
num_iterations = 3

for iteration in range(num_iterations):
    X_all = df_features[feature_cols]
    y_all = df_features['value_filled']
    
    refined_model = CatBoostRegressor(
        **best_params,
        loss_function='RMSE',
        task_type='GPU',
        devices='0',
        verbose=False
    )
    all_pool = Pool(X_all, y_all, cat_features=cat_feature_indices)
    refined_model.fit(all_pool)
    
    df_features.loc[df_missing.index, 'value_imputed'] = refined_model.predict(X_missing)
    df_features.loc[df_features['value'].isna(), 'value_filled'] = df_features['value_imputed']
    
    change = np.abs(df_features.loc[df_missing.index, 'value_imputed'] - prev_imputed_values).mean()
    if change < 0.001:
        break
    prev_imputed_values = df_features.loc[df_missing.index, 'value_imputed'].copy()

# Save results
output_df = df_features[['series_id', 'class', 'source', 'time', 'value', 'value_filled']]
output_df.to_csv("C:\\Users\\Graeme\\Documents\\github\\tsfwpt\\data\\dodgerloopday_imputed.csv", index=False)

# Plot original vs imputed distributions
plt.hist(df_features[df_features['value'].notna()]['value'], bins=50, alpha=0.5, label='Original')
plt.hist(df_features[df_features['value'].isna()]['value_imputed'], bins=50, alpha=0.5, label='Imputed')
plt.legend()
plt.title("Original vs Imputed Value Distributions")
plt.xlabel("Traffic count")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()